# RL

This notebook runs the full research flow using Python function calls only (no shell commands).

## 0) Environment and paths

In [45]:
from pathlib import Path
import csv
import json
import os

ROOT = Path.cwd()
if not (ROOT / 'coup').exists():
    fallback = Path('/Users/crishuynh/Documents/SoftwareProject/coup_detection')
    if (fallback / 'coup').exists():
        ROOT = fallback
os.chdir(ROOT)
print('Repository root:', ROOT)

SIM_DIR = ROOT / 'data' / 'sim'
RESEARCH_DIR = ROOT / 'data' / 'research'
MARKOV_CSV = RESEARCH_DIR / 'markov_transitions.csv'
BC_DIR = RESEARCH_DIR / 'bc'
DQN_DIR = RESEARCH_DIR / 'dqn_demo'
DATASET_CSV = RESEARCH_DIR / 'research_dataset.csv'
MODELS_DIR = RESEARCH_DIR / 'models'
REPORT_DIR = RESEARCH_DIR / 'report'
PLOTS_DIR = RESEARCH_DIR / 'plots'

SIM_DIR.mkdir(parents=True, exist_ok=True)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

Repository root: /Users/crishuynh/Documents/SoftwareProject/coup_detection


## 1) Configure experiment

In [46]:
CONFIG = {
    'players': 4,
    'games': 50,
    'seed': 123,
    'max_turns': 200,
    'bot_mix': None,  # example: ['honest', 'bluffer', 'aggressive', 'cautious']
    'write_trace': True,
    'dataset_horizon': 5,
    'dqn_episodes': 5000,
}
CONFIG

{'players': 4,
 'games': 50,
 'seed': 123,
 'max_turns': 200,
 'bot_mix': None,
 'write_trace': True,
 'dataset_horizon': 5,
 'dqn_episodes': 5000}

## 2) Simulate games and save `game_<n>.json`, optional `trace_<n>.json`, and `summary.csv`

In [47]:
from coup.sim.metrics import build_trace, calibration_bin_labels, summarize_game
from coup.sim.sim import simulate_games

results = simulate_games(
    CONFIG['games'],
    players=CONFIG['players'],
    seed=CONFIG['seed'],
    bot_mix=CONFIG['bot_mix'],
    max_turns=CONFIG['max_turns'],
)

histogram_columns = calibration_bin_labels()
summary_rows = []

for idx, result in enumerate(results, start=1):
    (SIM_DIR / f'game_{idx}.json').write_text(json.dumps(result.events, indent=2))

    if CONFIG['write_trace']:
        trace = build_trace(result.events, result.players)
        (SIM_DIR / f'trace_{idx}.json').write_text(json.dumps(trace, indent=2))

    metrics = summarize_game(
        events=result.events,
        players=result.players,
        winner=result.winner,
        turns=result.turns,
    )
    row = {
        'game': idx,
        'seed': CONFIG['seed'] + idx - 1,
        'players': len(result.players),
        'bot_mix': ','.join(result.bot_styles),
        'winner': metrics.winner,
        'turns': metrics.turns,
        'challenges': metrics.challenges,
        'challenge_accuracy': round(metrics.challenge_accuracy, 4),
        'advisor_agreement_rate': round(metrics.advisor_agreement_rate, 4),
    }
    for key in histogram_columns:
        row[f'reveal_prob_bin_{key}'] = metrics.reveal_prob_histogram.get(key, 0)
    summary_rows.append(row)

summary_csv = SIM_DIR / 'summary.csv'
if summary_rows:
    with summary_csv.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)

print(f'Saved {len(results)} games to {SIM_DIR}')
print(f'Summary CSV: {summary_csv}')
summary_rows[:2]

Saved 50 games to /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/sim
Summary CSV: /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/sim/summary.csv


[{'game': 1,
  'seed': 123,
  'players': 4,
  'bot_mix': 'honest,bluffer,aggressive,cautious',
  'winner': 'P1',
  'turns': 26,
  'challenges': 4,
  'challenge_accuracy': 0.0,
  'advisor_agreement_rate': 0.6667,
  'reveal_prob_bin_0.0-0.1': 1,
  'reveal_prob_bin_0.1-0.2': 0,
  'reveal_prob_bin_0.2-0.3': 2,
  'reveal_prob_bin_0.3-0.4': 0,
  'reveal_prob_bin_0.4-0.5': 1,
  'reveal_prob_bin_0.5-0.6': 0,
  'reveal_prob_bin_0.6-0.7': 0,
  'reveal_prob_bin_0.7-0.8': 0,
  'reveal_prob_bin_0.8-0.9': 0,
  'reveal_prob_bin_0.9-1.0': 2},
 {'game': 2,
  'seed': 124,
  'players': 4,
  'bot_mix': 'honest,bluffer,aggressive,cautious',
  'winner': 'P4',
  'turns': 26,
  'challenges': 6,
  'challenge_accuracy': 0.5,
  'advisor_agreement_rate': 0.5484,
  'reveal_prob_bin_0.0-0.1': 1,
  'reveal_prob_bin_0.1-0.2': 1,
  'reveal_prob_bin_0.2-0.3': 0,
  'reveal_prob_bin_0.3-0.4': 1,
  'reveal_prob_bin_0.4-0.5': 1,
  'reveal_prob_bin_0.5-0.6': 0,
  'reveal_prob_bin_0.6-0.7': 0,
  'reveal_prob_bin_0.7-0.8': 0,

## 3) Build Markov transition CSV

In [48]:
from coup.research.markov import build_transition_dataset_from_sim_dir, write_transition_csv

transition_rows = build_transition_dataset_from_sim_dir(SIM_DIR)
write_transition_csv(transition_rows, MARKOV_CSV)
print('Transitions:', len(transition_rows))
print('CSV:', MARKOV_CSV)

Transitions: 1223
CSV: /Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/markov_transitions.csv


## 4) Train behavior cloning and benchmark vs random

In [49]:
from coup.research.bc import benchmark_policy_vs_random, train_behavior_clone

bc_train_metrics = train_behavior_clone(MARKOV_CSV, BC_DIR, seed=42)
bc_benchmark = benchmark_policy_vs_random(MARKOV_CSV, BC_DIR, seed=42)

print('BC train metrics:')
print(json.dumps(bc_train_metrics, indent=2))
print('\nBC benchmark:')
print(json.dumps(bc_benchmark, indent=2))

BC train metrics:
{
  "accuracy": 0.369281045751634,
  "f1_macro": 0.23485094038242793
}

BC benchmark:
{
  "behavior_cloning": {
    "action_match_rate": 0.39084219133278825,
    "avg_reward_per_step": 0.8503679476696647,
    "avg_reward_per_episode": 0.8503679476696647
  },
  "random_policy": {
    "action_match_rate": 0.15290269828291086,
    "avg_reward_per_step": 0.6124284546197875,
    "avg_reward_per_episode": 0.6124284546197875
  }
}


## 5) Train DQN and benchmark vs random (+ BC comparison)

In [50]:
from coup.research.rl_train import train_dqn_policy

dqn_summary = train_dqn_policy(
    transitions_csv=MARKOV_CSV,
    out_dir=DQN_DIR,
    episodes=CONFIG['dqn_episodes'],
    seed=42,
    compare_bc_dir=BC_DIR,
)

print(json.dumps(dqn_summary, indent=2))

{
  "train_metrics": {
    "episodes": 5000,
    "avg_episode_reward": 22.901,
    "avg_episode_steps": 24.46,
    "final_epsilon": 0.05
  },
  "benchmark": {
    "dqn_policy": {
      "avg_reward_per_episode": 25.86,
      "avg_reward_per_step": 1.0572363041700736,
      "action_match_rate": 0.5977105478331971
    },
    "random_policy": {
      "avg_reward_per_episode": 15.16,
      "avg_reward_per_step": 0.6197874080130826,
      "action_match_rate": 0.16026165167620604
    },
    "behavior_cloning": {
      "action_match_rate": 0.39084219133278825,
      "avg_reward_per_step": 0.8503679476696647,
      "avg_reward_per_episode": 0.8503679476696647
    }
  }
}


## 6) Compare DQN vs Random vs BC metrics

In [51]:
import pandas as pd

bench = dqn_summary.get('benchmark', {})
rows = []
for policy_name in ['dqn_policy', 'random_policy', 'behavior_cloning']:
    if policy_name not in bench:
        continue
    item = bench[policy_name]
    rows.append({
        'policy': policy_name,
        'episodes': item.get('episodes'),
        'total_steps': item.get('total_steps'),
        'total_reward': item.get('total_reward'),
        'avg_reward_per_episode': item.get('avg_reward_per_episode'),
        'avg_reward_per_step': item.get('avg_reward_per_step'),
        'action_match_rate': item.get('action_match_rate'),
    })

pd.DataFrame(rows).sort_values('avg_reward_per_episode', ascending=False)



,policy,episodes,total_steps,total_reward,avg_reward_per_episode,avg_reward_per_step,action_match_rate
0,dqn_policy,None,None,None,25.860000,1.057236,0.597711
1,random_policy,None,None,None,15.160000,0.619787,0.160262
2,behavior_cloning,None,None,None,0.850368,0.850368,0.390842


## 7) Optional supervised dataset/report/plots (all Python API calls)

In [52]:
from coup.research.dataset import build_dataset_from_sim_dir, write_dataset_csv
from coup.research.plot import generate_plots
from coup.research.report import generate_report
from coup.research.train import train_baseline

dataset_rows = build_dataset_from_sim_dir(SIM_DIR, horizon=CONFIG['dataset_horizon'])
write_dataset_csv(dataset_rows, DATASET_CSV)
baseline_metrics = train_baseline(DATASET_CSV, MODELS_DIR, seed=42)
report = generate_report(DATASET_CSV, MODELS_DIR, REPORT_DIR, calibration_bins=10)
plot_paths = generate_plots(REPORT_DIR, PLOTS_DIR)

print('Dataset rows:', len(dataset_rows))
print('Baseline targets:', list(baseline_metrics.keys()))
print('Report targets:', list(report.keys()))
print('Generated plots:', [str(p) for p in plot_paths[:4]])

Dataset rows: 2997
Baseline targets: ['y_next_action', 'y_next_is_challenge', 'y_coup_within_horizon', 'y_current_claim_challenged']
Report targets: ['y_next_action', 'y_next_is_challenge', 'y_coup_within_horizon', 'y_current_claim_challenged']
Generated plots: ['/Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/plots/accuracy_by_target.png', '/Users/crishuynh/Documents/SoftwareProject/coup_detection/data/research/plots/f1_by_target.png']


## 8) Reproducibility checklist
- Keep `CONFIG` fixed (players, games, seed, bot mix, dqn episodes).
- Run the full notebook from top to bottom.
- Save `data/sim/`, `data/research/dqn_demo/summary.json`, and `data/research/report/metrics_table.csv` as thesis evidence.